In [ ]:
from google.colab import files
uploaded=files.upload()

Saving TASK 2.sql to TASK 2 (1).sql


In [ ]:
# task 3.1

import pandas as pd
from sklearn.model_selection import train_test_split

df=pd.read_csv("/content/appointments_clean.csv")
features=["age","distance_km","fee_inr","lead_days","prior_no_shows","reminder_sent","booking_channel","visit_type"]
x=df[features].copy()
x["reminder_sent"]=x["reminder_sent"].map({"Yes":1,"No":0})
x=pd.get_dummies(x,columns=["booking_channel","visit_type"],dtype=int)
y=df["no_show"]
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)
print("x columns:",x.shape[1],list(x.columns))
print("train rows:",len(x_train),"test rows:",len(x_test))
print("no shows in test set:",y_test.sum(),"(%.1f%% of the test set)"%(y_test.mean()*100))
baseline=(y_test==0).mean()
print("baseline accuracy:%.4f"%baseline)

x columns: 11 ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']
train rows: 792 test rows: 198
no shows in test set: 49 (24.7% of the test set)
baseline accuracy:0.7525


In [ ]:
#2 logistic regression

import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score,confusion_matrix,precision_score,recall_score,f1_score,roc_auc_score)

scaler=StandardScaler()
x_train_s=scaler.fit_transform(x_train)
x_test_s=scaler.transform(x_test)

model=LogisticRegression(max_iter=1000)
model.fit(x_train_s,y_train)

pred=model.predict(x_test_s)
proba=model.predict_proba(x_test_s)[:,1]

print("accuracy:",round(accuracy_score(y_test,pred),3))
print("confusion matrix[[TN,FP],[FN,TP]]:\n",confusion_matrix(y_test,pred))
print("precision:",round(precision_score(y_test,pred),2))
print("recall:",round(recall_score(y_test,pred),2))
print("F1:",round(f1_score(y_test,pred),2))
print("ROC-AUC:",round(roc_auc_score(y_test,proba),3))

coefs=pd.Series(model.coef_[0],index=x.columns).sort_values()
print("\n coefficients(scaled features):")
print(coefs.round(2).to_string())

i=list(x.columns).index("reminder_sent")
coef=model.coef_[0][i]
scale=scaler.scale_[i]
print("\n reminder sent coefficients:",round(coef,2),"SD of the features:",round(scale,2))
print("coefficient=",round(np.exp(coef),2),"(oddds multiplier per ONE SD)")
print("coefficient/scale=",round(np.exp(coef/scale),2),"(oddds multiplier for NO---YES)")




accuracy: 0.788
confusion matrix[[TN,FP],[FN,TP]]:
 [[139  10]
 [ 32  17]]
precision: 0.63
recall: 0.35
F1: 0.45
ROC-AUC: 0.789

 coefficients(scaled features):
reminder_sent             -0.58
age                       -0.25
booking_channel_Walk-In   -0.19
booking_channel_App       -0.05
visit_type_Follow-up      -0.02
visit_type_Consultation    0.02
booking_channel_Phone      0.22
distance_km                0.25
fee_inr                    0.45
lead_days                  0.61
prior_no_shows             0.65

 reminder sent coefficients: -0.58 SD of the features: 0.49
coefficient= 0.56 (oddds multiplier per ONE SD)
coefficient/scale= 0.31 (oddds multiplier for NO---YES)


In [ ]:
# task 3.3
from sklearn.tree import DecisionTreeClassifier

tree_full=DecisionTreeClassifier(random_state=42).fit(x_train,y_train)
tree3=DecisionTreeClassifier(max_depth=3,random_state=42).fit(x_train,y_train)

for name,t in[("no depth limit" ,tree_full),("max_depth=3",tree3)]:
  print(name,"depth:",t.get_depth(),
        "training accuracy:%.2f"%t.score(x_train,y_train),
         "test accuracy:%.2f"%t.score(x_test,y_test) )

print("baseline to beat:%.4f"%baseline)





no depth limit depth: 15 training accuracy:1.00 test accuracy:0.67
max_depth=3 depth: 3 training accuracy:0.82 test accuracy:0.78
baseline to beat:0.7525


In [ ]:
# task 3.4
from sklearn.ensemble import RandomForestClassifier
rf=RandomForestClassifier(n_estimators=100,random_state=42).fit(x_train,y_train)
rf_pred=rf.predict(x_test)
rf_proba=rf.predict_proba(x_test)[:,1]
print("train_accuracy:%.2f" %rf.score(x_train,y_train))
print("test accuracy:%.2f" %accuracy_score(y_test,rf_pred))
print("recall:%.2f" %recall_score(y_test,rf_pred))
print("ROC-AUC:%.2f" %roc_auc_score(y_test,rf_proba))

importances=pd.Series(rf.feature_importances_,index=x.columns).sort_values(ascending=False)
print("\n top three importances:")
print(importances.head(3).round(2).to_string())

# comparison table
tree3_pred=tree3.predict(x_test)
tree3_proba=tree3.predict_proba(x_test)[:,1]
comparison=pd.DataFrame({"accuracy":[accuracy_score(y_test,pred),accuracy_score(y_test,tree3_pred),accuracy_score(y_test,rf_pred)],
                         "recall":[recall_score(y_test,pred),recall_score(y_test,tree3_pred),recall_score(y_test,rf_pred)],
                         "ROC-AUC":[roc_auc_score(y_test,proba),roc_auc_score(y_test,tree3_proba),roc_auc_score(y_test,rf_proba)],},index=["Logistic regression","Decision tree (depth 3)","Random Forest"]).round(2)

print("\n",comparison)






train_accuracy:1.00
test accuracy:0.75
recall:0.31
ROC-AUC:0.71

 top three importances:
lead_days      0.22
age            0.20
distance_km    0.19

                          accuracy  recall  ROC-AUC
Logistic regression          0.79    0.35     0.79
Decision tree (depth 3)      0.78    0.22     0.70
Random Forest                0.75    0.31     0.71


In [ ]:
# TASK 3.5
from sklearn.model_selection import GridSearchCV,cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

#1 TUNE THE FOREST IN THE TRAINING SET
grid=GridSearchCV(RandomForestClassifier(random_state=42),{"n_estimators":[100,200],"max_depth":[3,5,8]},
                                       cv=5,scoring="roc_auc")
grid.fit(x_train,y_train)
print("params:",grid.best_params_)
print("score_(cross validated ROC-AUC):%.3f"%grid.best_score_)

#2 5-FOLD CV  ON TRAINING ET
lr_pipeline=Pipeline([("scaler",StandardScaler()),("model",LogisticRegression(max_iter=1000))])
cv_scores={
          "logistic regression":cross_val_score(lr_pipeline,x_train,y_train,cv=5, scoring="roc_auc").mean(),
          "decision trree (depth 3)":cross_val_score(DecisionTreeClassifier(max_depth=3,random_state=42),x_train,y_train,cv=5, scoring="roc_auc").mean(),
          "random forest ":cross_val_score(RandomForestClassifier(n_estimators=100,random_state=42),x_train,y_train,cv=5, scoring="roc_auc").mean(),
          "tuned random forest": grid.best_score_,
         }

for name, score in cv_scores.items():
  print("%-26s CV ROC-AUC=%.3f"%(name,score))
winner=max(cv_scores,key=cv_scores.get)
print("Winner(highest CV ROC-AUC):", winner)

#3 THRESHOLD USING OUT OF FIELD POBABAILITIES ON TRAINING SET
from sklearn.model_selection import cross_val_predict,cross_val_score
from sklearn.metrics import precision_score,recall_score,f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

lr_pipe=Pipeline([("scaler",StandardScaler()),("model",LogisticRegression(max_iter=1000))])
lr_pipe.fit(x_train,y_train)

oof=cross_val_predict(lr_pipe,x_train,y_train,cv=5,method="predict_proba")[:,1]
rows=[]
for t in (0.5,0.35):
  flag=(oof>=t).astype(int)
  rows.append({
              "threshold":t,
              "precision":precision_score(y_train,flag),
              "recall":recall_score(y_train,flag),
              "F1":f1_score(y_train,flag),
              "FLAGGED":int(flag.sum())
  })

thr_table=pd.DataFrame(rows).round(2)
print(thr_table.to_string(index=False))
chosen=thr_table.loc[thr_table["F1"].idxmax(),"threshold"]
print("chosen threshold (higher f1):",chosen)

#4 evaluate on the test set
lr_pipe.fit(x_train,y_train)
probs=lr_pipe.predict_proba(x_test)[:,1]
flag_test=(probs>=chosen).astype((int))
print("threshold:",chosen)
print("precision:%.2f"%precision_score(y_test,flag_test))
print("recall:%.2f"%recall_score(y_test,flag_test))
print("flagged:",int(flag_test.sum()),"of",len(y_test))
print("ROC-AUC:%.2f"%roc_auc_score(y_test,probs))

# save the scores
out=pd.DataFrame({
                  "appointment_id":df.loc[x_test.index,"appointment_id"].values,
                  "no_show":y_test.values,
                  "risk_prob":probs.round(4),
                  "flag":flag_test
                  })
out.to_csv("test_risk_scores.csv",index=False)
print("saved test_risk_scores.csv:",out.shape,"rows with flag =1",int(out["flag"].sum()))







params: {'max_depth': 3, 'n_estimators': 200}
score_(cross validated ROC-AUC):0.767
logistic regression        CV ROC-AUC=0.774
decision trree (depth 3)   CV ROC-AUC=0.678
random forest              CV ROC-AUC=0.747
tuned random forest        CV ROC-AUC=0.767
Winner(highest CV ROC-AUC): logistic regression
 threshold  precision  recall   F1  FLAGGED
      0.50       0.58    0.25 0.35       73
      0.35       0.51    0.47 0.49      154
chosen threshold (higher f1): 0.35
threshold: 0.35
precision:0.57
recall:0.57
flagged: 49 of 198
ROC-AUC:0.79
saved test_risk_scores.csv: (198, 4) rows with flag =1 49
